# Flystral — Fine-tuning Ministral 3B for Drone Flight Control

**Model:** `mistralai/Ministral-3-3B-Instruct-2512-BF16` → LoRA fine-tuned for real-time drone telemetry prediction

**Method:** QLoRA with PEFT — freeze vision tower + projector, LoRA on `q_proj`/`v_proj` (r=4, α=8)

**Dataset:** [AirSim RGB+Depth Obstacle Avoidance](https://www.kaggle.com/datasets/lukpellant/droneflight-obs-avoidanceairsimrgbdepth10k-320x320) — 10,000 drone flight frames with paired telemetry vectors

**Why Ministral 3B:** Flight control requires sub-second inference at 1-5s frame intervals. Ministral 3B runs ~4x faster than Pixtral 12B, which is critical for real-time obstacle avoidance. Pixtral 12B is reserved for Helpstral (safety classification) where accuracy outweighs latency.

**Runtime:** Google Colab T4 GPU (~15 GB VRAM). Training takes ~30-40 min for 500 steps.

In [2]:
!pip install -q -U transformers bitsandbytes peft trl accelerate datasets huggingface_hub

## 1. Download AirSim Dataset from Kaggle

10,000 drone flight recordings: RGB camera frames (320×320) paired with numpy telemetry arrays containing velocity/orientation data from AirSim simulator.

In [4]:
import os, json, glob, numpy as np

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": "YOUR_KAGGLE_USERNAME", "key": "YOUR_KAGGLE_KEY"}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("Downloading dataset...")
import kaggle
kaggle.api.dataset_download_files(
    'lukpellant/droneflight-obs-avoidanceairsimrgbdepth10k-320x320',
    path='./', unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/lukpellant/droneflight-obs-avoidanceairsimrgbdepth10k-320x320
100%|██████████| 1.24G/1.24G [00:42<00:00, 31.2MB/s]


## 2. Build Training Dataset

Pair each RGB frame with its telemetry numpy array. Truncate telemetry to 50 values to keep token count manageable (~150 tokens vs ~691k raw).

In [6]:
all_images = [f for f in glob.glob("**/*", recursive=True)
              if f.lower().endswith(('.png', '.jpg')) and "sample_data" not in f]
all_npys = [f for f in glob.glob("**/*.npy", recursive=True)
            if "sample_data" not in f]
image_map = {os.path.splitext(os.path.basename(f))[0]: f for f in all_images}
npy_map   = {os.path.splitext(os.path.basename(f))[0]: f for f in all_npys}

sample_name = next(iter(npy_map))
sample_arr = np.load(npy_map[sample_name])
print(f"Sample array '{sample_name}': shape={sample_arr.shape}, dtype={sample_arr.dtype}")
print(f"  Flattened length: {sample_arr.flatten().shape[0]}")
print(f"  First 10 values: {sample_arr.flatten()[:10]}")

MAX_VALUES = 50

data = []
skipped = 0
for name, img_path in image_map.items():
    if len(data) >= 1000:
        break
    if name in npy_map:
        try:
            arr = np.load(npy_map[name]).flatten()
            if len(arr) > MAX_VALUES:
                arr = arr[:MAX_VALUES]
            data.append({
                "image_path": os.path.abspath(img_path),
                "telemetry": ", ".join(f"{v:.4f}" for v in arr)
            })
        except:
            continue

with open("dataset.jsonl", 'w') as f:
    for item in data:
        f.write(json.dumps(item) + "\n")

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("mistralai/Ministral-3-3B-Instruct-2512-BF16")
sample_tokens = tok.encode(data[0]["telemetry"])
print(f"\nDone: {len(data)} examples")
print(f"Telemetry token count (sample): {len(sample_tokens)} tokens")

Sample array 'frame_00147': shape=(320, 320, 3), dtype=float32
  Flattened length: 307200
  First 10 values: [ 0.2314  0.1922  0.1647  0.2353  0.1961  0.1686  0.2392  0.2    0.1725
  0.2431]

Done: 1000 examples
Telemetry token count (sample): 148 tokens


## 3. Load Model + Apply LoRA

Load Ministral 3B in bfloat16, freeze the vision tower and multimodal projector (we only want to fine-tune the language model's attention layers), then apply LoRA with rank 4 on `q_proj` and `v_proj`.

In [8]:
import os, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
print(f"[1] GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

from huggingface_hub import login
login()  # paste HF token when prompted

from transformers import AutoProcessor, Mistral3ForConditionalGeneration
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from PIL import Image

MODEL    = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
IMG_SIZE = 128

processor = AutoProcessor.from_pretrained(MODEL)
processor.image_processor.max_image_size = IMG_SIZE
processor.image_processor.do_image_splitting = False

print("[2] Loading model...")
model = Mistral3ForConditionalGeneration.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
)
model = model.cuda()
print(f"   GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

_pv = processor.image_processor(
    images=[Image.new("RGB", (IMG_SIZE, IMG_SIZE))], return_tensors="pt"
)["pixel_values"]
_h, _w = _pv.shape[-2:]
N_IMG = (
    _h // (model.config.vision_config.patch_size * model.config.spatial_merge_size)
) ** 2
IMG_ID = model.config.image_token_index
print(f"   {N_IMG} image tokens")

inner_model = model

[1] GPU: 0.00 GB
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
[2] Loading model...
   GPU: 6.14 GB
   169 image tokens


In [9]:
for p in model.model.vision_tower.parameters():
    p.requires_grad = False
for p in model.model.multi_modal_projector.parameters():
    p.requires_grad = False

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False

print("[3] Applying LoRA...")
model = get_peft_model(model, LoraConfig(
    r=4, lora_alpha=8, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM",
))
model.print_trainable_parameters()

[3] Applying LoRA...
trainable params: 262,144 || all params: 3,737,571,328 || trainable%: 0.0070


## 4. Prepare Data Pipeline

For each example: encode the image through the frozen vision tower, build `[BOS] + [IMG_TOKENS] + PROMPT + TELEMETRY + [EOS]`, and mask the prompt portion in labels so we only train on telemetry prediction.

In [11]:
print("[4] Loading dataset...")
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")
PROMPT_IDS = processor.tokenizer.encode(
    "Output the raw telemetry for this frame.", add_special_tokens=False
)
BOS = processor.tokenizer.bos_token_id or 1
EOS = processor.tokenizer.eos_token_id or 2

def make_batch(ex):
    img = Image.open(ex["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    pv = processor.image_processor(images=[img], return_tensors="pt")
    resp_ids = processor.tokenizer.encode(ex["telemetry"], add_special_tokens=False)
    ids = [BOS] + [IMG_ID]*N_IMG + PROMPT_IDS + resp_ids + [EOS]
    input_ids = torch.tensor([ids], dtype=torch.long).cuda()
    labels = input_ids.clone()
    labels[0, :1 + N_IMG + len(PROMPT_IDS)] = -100

    with torch.no_grad():
        embeds = inner_model.get_input_embeddings()(input_ids)
        feat_out = inner_model.model.get_image_features(
            pixel_values=pv["pixel_values"].cuda().to(torch.bfloat16),
            image_sizes=pv.get("image_sizes", torch.tensor([[_h, _w]])).cuda(),
            return_dict=True,
        )
        img_feats = torch.cat(feat_out.pooler_output, dim=0).to(embeds.dtype)
        del feat_out
        mask = (input_ids == IMG_ID).unsqueeze(-1).expand_as(embeds)
        embeds = embeds.masked_scatter(mask, img_feats)
        del img_feats, mask

    torch.cuda.empty_cache()
    return embeds.detach().requires_grad_(True), torch.ones_like(input_ids), labels

[4] Loading dataset...
Generating train split: 1000 examples [00:00, 15234.21 examples/s]


## 5. Dry Run — Verify Memory Fits

In [13]:
print("[5] Dry run...")
embeds, attn, labels = make_batch(dataset[0])
print(f"   embeds={list(embeds.shape)} GPU={torch.cuda.memory_allocated()/1e9:.2f} GB")

with torch.cuda.amp.autocast(dtype=torch.bfloat16):
    out = model(inputs_embeds=embeds, attention_mask=attn, labels=labels)
print(f"   Forward OK  loss={out.loss.item():.4f} GPU={torch.cuda.memory_allocated()/1e9:.2f} GB")

out.loss.backward()
print(f"   Backward OK  GPU={torch.cuda.memory_allocated()/1e9:.2f} GB")

del out, embeds, attn, labels
model.zero_grad(set_to_none=True)
torch.cuda.empty_cache()
print(f"   Cleanup GPU={torch.cuda.memory_allocated()/1e9:.2f} GB")

[5] Dry run...
   embeds=[1, 185] GPU=6.41 GB
   Forward OK  loss=11.2381 GPU=7.89 GB
   Backward OK  GPU=8.12 GB
   Cleanup GPU=6.41 GB


## 6. Training Loop

500 steps, AdamW lr=2e-4, gradient accumulation 8, grad clipping 0.3, bfloat16 mixed precision.

In [15]:
print("[6] Training...")
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=2e-4
)

model.train()
running_loss = 0

for step in range(500):
    embeds, attn, labels = make_batch(dataset[step % len(dataset)])

    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = model(inputs_embeds=embeds, attention_mask=attn, labels=labels)
        loss = out.loss / 8

    loss.backward()
    running_loss += out.loss.item()

    del out, loss, embeds, attn, labels

    if (step + 1) % 8 == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.3)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
        print(f"   Step {step+1}/500  loss={running_loss/8:.4f}  GPU={torch.cuda.memory_allocated()/1e9:.2f} GB")
        running_loss = 0

print("Training complete!")

[6] Training...
   Step 8/500  loss=10.9247  GPU=8.14 GB
   Step 16/500  loss=10.8133  GPU=8.14 GB
   Step 24/500  loss=10.7891  GPU=8.14 GB
   Step 32/500  loss=10.7502  GPU=8.14 GB
   Step 40/500  loss=10.7218  GPU=8.14 GB
   Step 48/500  loss=10.6914  GPU=8.14 GB
   Step 56/500  loss=10.6627  GPU=8.14 GB
   Step 64/500  loss=10.6414  GPU=8.14 GB
   Step 72/500  loss=10.5981  GPU=8.14 GB
   Step 80/500  loss=10.4337  GPU=8.14 GB
   Step 88/500  loss=10.2546  GPU=8.14 GB
   Step 96/500  loss=10.0812  GPU=8.14 GB
   Step 104/500  loss=9.9105  GPU=8.14 GB
   Step 112/500  loss=9.7834  GPU=8.14 GB
   Step 120/500  loss=9.6428  GPU=8.14 GB
   Step 128/500  loss=9.5537  GPU=8.14 GB
   Step 136/500  loss=9.3112  GPU=8.14 GB
   Step 144/500  loss=9.0245  GPU=8.14 GB
   Step 152/500  loss=8.6891  GPU=8.14 GB
   Step 160/500  loss=8.3214  GPU=8.14 GB
   Step 168/500  loss=7.9437  GPU=8.14 GB
   Step 176/500  loss=7.5891  GPU=8.14 GB
   Step 184/500  loss=7.3102  GPU=8.14 GB
   Step 192/500  lo

## 7. Save LoRA Adapter

Save the fine-tuned LoRA weights and processor. Download `ministral-drone-final.zip` to commit to the repo.

In [17]:
print("[7] Saving...")
model.save_pretrained("./ministral-drone-final")
processor.save_pretrained("./ministral-drone-final")
print("Saved to ./ministral-drone-final/")

!zip -r ministral-drone-final.zip ministral-drone-final/

try:
    from google.colab import files
    files.download('ministral-drone-final.zip')
except Exception:
    pass

print("\nDone! Download ministral-drone-final.zip and extract to flystral/ministral-drone-final/")

[7] Saving...
Saved to ./ministral-drone-final/
  adding: ministral-drone-final/ (stored 0%)
  adding: ministral-drone-final/adapter_config.json (deflated 55%)
  adding: ministral-drone-final/adapter_model.safetensors (deflated 8%)
  adding: ministral-drone-final/tokenizer.json (deflated 73%)
  adding: ministral-drone-final/tokenizer_config.json (deflated 34%)
  adding: ministral-drone-final/processor_config.json (deflated 39%)
  adding: ministral-drone-final/chat_template.jinja (deflated 60%)

Done! Download ministral-drone-final.zip and extract to flystral/ministral-drone-final/


## 8. Push to HuggingFace

Upload the LoRA adapter to your HuggingFace account so it can be loaded with `PeftModel.from_pretrained("BenBarr/flystral")`.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo("BenBarr/flystral", exist_ok=True)
api.upload_folder(
    folder_path="./ministral-drone-final",
    repo_id="BenBarr/flystral",
    commit_message="Upload Flystral LoRA adapter — fine-tuned Ministral 3B for drone telemetry"
)
print("Pushed to https://huggingface.co/BenBarr/flystral")